In [1]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl accelerate

import os, json, re, time, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
import emoji
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.7 MB/s eta 0:00:00
Device: cuda


In [2]:
REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

with open(os.path.join(DOCS_PATH, "patterns.json"), encoding="utf-8") as f:
    pattern_dict = json.load(f)
with open(os.path.join(DOCS_PATH, "emojis.json"), encoding="utf-8") as f:
    emoji_dict = json.load(f)

teen_dict = {}
with open(os.path.join(DOCS_PATH, "teencode4.txt"), encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and "\t" in line:
            old, new = line.split("\t", 1)
            teen_dict[old] = new

excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
excel_file = pd.ExcelFile(excel_path)
if "train" in excel_file.sheet_names:
    train_df = pd.read_excel(excel_file, sheet_name="train")
    val_df   = pd.read_excel(excel_file, sheet_name="val")
    test_df  = pd.read_excel(excel_file, sheet_name="test")
else:
    df = pd.read_excel(excel_file, sheet_name="Sheet1")
    train_df = df[df["set"] == "train"].copy()
    val_df   = df[df["set"] == "val"].copy()
    test_df  = df[df["set"] == "test"].copy()

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

Cloning into '/kaggle/working/ViGoEmotions_Original'...


Train: (16531, 3) | Val: (2066, 3) | Test: (2067, 3)


In [3]:
def normalize_pattern(text):
    for pattern, replacement in pattern_dict.items():
        text = re.sub(pattern=pattern, repl=replacement, string=text)
    return text

def remove_duplicate_chars(text):
    prev_char, result = None, []
    for char in text:
        if char.isalpha() and prev_char == char:
            continue
        prev_char = char
        result.append(char)
    return "".join(result)

def remove_duplicate_emoji(text):
    result, prev_emoji = [], None
    for char in text:
        if char in emoji.EMOJI_DATA:
            if char == prev_emoji:
                continue
            prev_emoji = char
        else:
            prev_emoji = None
        result.append(char)
    return "".join(result)

def replace_teencode(text):
    for old_word, new_word in teen_dict.items():
        pattern = re.compile(r"\b{}\b".format(re.escape(old_word)))
        text = pattern.sub(new_word, text)
    return text

def clean_text(text):
    """S1 theo thầy: KHÔNG gọi replacing_emojis"""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = normalize_pattern(text)
    text = remove_duplicate_chars(text)
    text = remove_duplicate_emoji(text)
    text = replace_teencode(text)
    # text = replacing_emojis(text)  # ← thầy COMMENT, S1 không dùng

    text = re.sub(r"(?<![.,!?;:])\n", r". ", text)
    text = re.sub(r"\n([.,!?;:])?", r" \1", text)
    text = re.sub(r"([.,!?;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Applying S1 preprocessing (no emoji replace)...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("Done:", train_df["text"].iloc[0])

Applying S1 preprocessing (no emoji replace)...
Done: xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


In [4]:
label_dict_path = os.path.join(DOCS_PATH, "label_dict.json")
if not os.path.exists(label_dict_path):
    label_dict_path = os.path.join(CORPUS_PATH, "label_dict.json")

with open(label_dict_path, encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Labels:", len(label_dict))

def encode_labels(label_str, label_dict):
    labels = str(label_str).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)
    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]
print("Labels encoded")

Labels: 28
Labels encoded


In [5]:
model_type = "xlm-r"
model_name = "FacebookAI/xlm-roberta-base"
max_len = 200
BATCH_SIZE = 16   # OOM thì 8

tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = torch.tensor(np.array(labels), dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        text = tokenize(text)  # theo thầy

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoader ready (XLM-R - S1)")
print("Train batches:", len(train_loader))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DataLoader ready (XLM-R - S1)
Train batches: 1034


In [6]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_type, model_name):
        super().__init__()
        self.model_type = model_type
        config = AutoConfig.from_pretrained(
            model_name,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1,
        )
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(self.backbone.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        if "bartpho" in self.model_type:
            pooled = outputs.last_hidden_state[:, 0, :]
        else:
            pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(len(label_dict), model_type, model_name).to(device)
print("XLM-R loaded (S1)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLM-R loaded (S1)
Parameters: 278,065,180


In [7]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)
            if is_train:
                optimizer.zero_grad()
            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())
    y, p = np.vstack(all_y), np.vstack(all_p)
    return np.mean(losses), f1_score(y, p, average="macro", zero_division=0)

best_f1 = 0.0
history = {"epoch": [], "train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

print("Start training BARTpho (S1)...")
for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_f1 = run_epoch(model, train_loader, True)
    val_loss, val_f1 = run_epoch(model, val_loader, False)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_s1_best.pth")
        print(f"Saved best (Val F1={best_f1:.4f})")

pd.DataFrame(history).to_excel(f"/kaggle/working/reports/metrics_{model_type}_s1.xlsx", index=False)
print("Metrics saved:", f"/kaggle/working/reports/metrics_{model_type}_s1.xlsx")

Start training BARTpho (S1)...

Epoch 1/12


Train Loss: 1.0982 | Train F1: 0.2018
Val   Loss: 0.8679 | Val   F1: 0.3356
Saved best (Val F1=0.3356)

Epoch 2/12


Train Loss: 0.7961 | Train F1: 0.3603
Val   Loss: 0.7579 | Val   F1: 0.4110
Saved best (Val F1=0.4110)

Epoch 3/12


Train Loss: 0.6445 | Train F1: 0.4325
Val   Loss: 0.6938 | Val   F1: 0.4320
Saved best (Val F1=0.4320)

Epoch 4/12


Train Loss: 0.5379 | Train F1: 0.4903
Val   Loss: 0.6975 | Val   F1: 0.4637
Saved best (Val F1=0.4637)

Epoch 5/12


...

Train Loss: 0.2663 | Train F1: 0.6891
Val   Loss: 0.8526 | Val   F1: 0.5384
Saved best (Val F1=0.5384)

Epoch 9/12


Train Loss: 0.2262 | Train F1: 0.7307
Val   Loss: 0.8926 | Val   F1: 0.5534
Saved best (Val F1=0.5534)

Epoch 10/12


Train Loss: 0.1936 | Train F1: 0.7661
Val   Loss: 0.9117 | Val   F1: 0.5619
Saved best (Val F1=0.5619)

Epoch 11/12


Train Loss: 0.1693 | Train F1: 0.7962
Val   Loss: 0.9604 | Val   F1: 0.5709
Saved best (Val F1=0.5709)

Epoch 12/12


Train Loss: 0.1502 | Train F1: 0.8172
Val   Loss: 0.9831 | Val   F1: 0.5754
Saved best (Val F1=0.5754)
Metrics saved: /kaggle/working/reports/metrics_xlm-r_s1.xlsx


In [8]:
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_s1_best.pth"))
model.eval()

all_targets, all_preds = [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)
        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()
        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
print(f"\nTEST Macro F1: {macro_f1:.4f}")
print(f"Test Micro F1: {micro_f1:.4f}")

report = classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0, output_dict=True
)
pd.DataFrame(report).transpose().to_excel(
    f"/kaggle/working/reports/classification_report_{model_type}_s1.xlsx",
    index=True
)
print(classification_report(y_true, y_pred, target_names=list(label_dict.values()), zero_division=0))
print("Report saved:", f"/kaggle/working/reports/classification_report_{model_type}_s1.xlsx")

100%|██████████| 130/130 [00:23<00:00,  5.50it/s]


TEST Macro F1: 0.5715
Test Micro F1: 0.5839
                precision    recall  f1-score   support

     amusement       0.66      0.85      0.74       374
    excitement       0.46      0.73      0.56        98
           joy       0.45      0.78      0.57       204
          love       0.52      0.87      0.65       143
        desire       0.36      0.68      0.47        80
      optimism       0.53      0.82      0.64       142
        caring       0.48      0.78      0.59       150
         pride       0.58      0.78      0.67        86
    admiration       0.46      0.71      0.56       101
     gratitude       0.76      0.88      0.82       108
        relief       0.41      0.83      0.55        60
      approval       0.51      0.76      0.61       115
   realization       0.37      0.55      0.44        95
      surprise       0.42      0.58      0.49        85
     curiosity       0.49      0.69      0.57       100
     confusion       0.37      0.58      0.45        84
  